# Hugging Face Governance Harness

Builds a **local cache** of model metadata once, then answers several audit
questions against that cache without re-crawling.

Supports two papers:

* **Paper 1** Governance controls do not propagate: license and use-restriction
  inheritance across model derivatives, with provenance completeness as the denominator.
* **Paper 2** Machine-readable versus human-readable model documentation:
  a contradiction audit of YAML frontmatter against card prose.

Runs on a free CPU Colab. No GPU. Stage A is one paginated API call.
Stage C is one small file download per model and is the only slow part.

**Reproducibility note.** Every stage writes to `CACHE` and is skipped if the
output already exists. Delete a cache file to force a re-run. `manifest.json`
records the configuration, package versions, timestamps, and row counts.

In [ ]:
# Cell 1: install and imports
# In Colab, uncomment the install line on first run.
# !pip -q install "huggingface_hub>=0.23" pandas matplotlib

import os
import re
import json
import time
import random
import hashlib
import pathlib
import datetime
import platform
from collections import Counter, defaultdict

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import huggingface_hub
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import (
    RepositoryNotFoundError,
    EntryNotFoundError,
    GatedRepoError,
    HfHubHTTPError,
)

print("huggingface_hub", huggingface_hub.__version__)
print("pandas", pd.__version__)

In [ ]:
# Cell 2: configuration
# Change these, then re-run. Nothing below this cell needs editing.

CONFIG = {
    "n_models": 3000,        # size of the bulk metadata sample (Stage A)
    "n_readme": 600,         # subset for the prose audit (Stage C), the slow stage
    "seed": 20261203,
    "sort_by": "downloads",  # top-N by downloads keeps the sample reproducible
    "readme_max_chars": 40000,
    "sleep_between_readmes": 0.15,   # be polite to the Hub
    "cache_dir": "/content/hf_gov_cache",  # use "./hf_gov_cache" outside Colab
}

# Optional: a read token raises your rate limit. Leave as None to run anonymously.
HF_TOKEN = None
# In Colab you can instead do:
# from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")

CACHE = pathlib.Path(CONFIG["cache_dir"])
CACHE.mkdir(parents=True, exist_ok=True)

MODELS_JSONL = CACHE / "models.jsonl"
PARENTS_JSONL = CACHE / "parents.jsonl"
README_JSONL = CACHE / "readmes.jsonl"
MANIFEST = CACHE / "manifest.json"

random.seed(CONFIG["seed"])
api = HfApi(token=HF_TOKEN)

print("cache:", CACHE.resolve())

In [ ]:
# Cell 3: small helpers

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def base_models_from_card(card_data):
    """Return declared base_model ids from cardData. Handles str and list forms."""
    if not isinstance(card_data, dict):
        return []
    raw = card_data.get("base_model")
    if raw is None:
        return []
    if isinstance(raw, str):
        raw = [raw]
    out = []
    for item in raw:
        if isinstance(item, str) and "/" in item:
            out.append(item.strip())
    return out


def base_models_from_tags(tags):
    """Fallback: the Hub also exposes relations as base_model:<kind>:<id> tags."""
    out = []
    for tag in tags or []:
        if not isinstance(tag, str) or not tag.startswith("base_model"):
            continue
        parts = tag.split(":")
        candidate = parts[-1]
        if "/" in candidate:
            out.append(candidate.strip())
    return out


def relation_kind_from_tags(tags):
    """finetune, adapter, quantized, merge, or unknown."""
    for tag in tags or []:
        if isinstance(tag, str) and tag.startswith("base_model:"):
            parts = tag.split(":")
            if len(parts) >= 3:
                return parts[1]
    return "unknown"


def license_from_tags(tags):
    for tag in tags or []:
        if isinstance(tag, str) and tag.startswith("license:"):
            return tag.split(":", 1)[1]
    return None


def card_to_dict(card):
    """cardData is a dict on hub<1.0 and a ModelCardData object on hub>=1.0."""
    if card is None:
        return None
    if isinstance(card, dict):
        return card
    for meth in ("to_dict", "dict"):
        if hasattr(card, meth):
            try:
                return getattr(card, meth)()
            except Exception:
                pass
    try:
        return dict(card)
    except Exception:
        return None


def normalize_model_info(info):
    """Flatten a ModelInfo into a plain dict. Attribute availability varies by
    huggingface_hub version, so every field is read defensively."""
    card = card_to_dict(getattr(info, "cardData", None) or getattr(info, "card_data", None))
    tags = list(getattr(info, "tags", []) or [])
    # hub>=1.0 exposes base_models natively; older versions need the card or tags
    native = [b for b in (getattr(info, "base_models", None) or []) if isinstance(b, str) and "/" in b]
    declared = native or base_models_from_card(card) or base_models_from_tags(tags)
    lic = None
    if isinstance(card, dict):
        lic = card.get("license")
    if lic is None:
        lic = getattr(info, "license", None)
    if lic is None:
        lic = license_from_tags(tags)
    if isinstance(lic, list):
        lic = lic[0] if lic else None

    return {
        "id": getattr(info, "id", None) or getattr(info, "modelId", None),
        "downloads": getattr(info, "downloads", None),
        "likes": getattr(info, "likes", None),
        "pipeline_tag": getattr(info, "pipeline_tag", None),
        "library_name": getattr(info, "library_name", None),
        "gated": getattr(info, "gated", None),
        "license_declared": lic,
        "base_models": declared,
        "relation_kind": relation_kind_from_tags(tags),
        "has_card_data": isinstance(card, dict) and len(card) > 0,
        "tags": tags,
        "last_modified": str(getattr(info, "lastModified", "") or getattr(info, "last_modified", "") or ""),
    }

In [ ]:
# Cell 4: STAGE A. Bulk metadata sample.
# One paginated listing call. Typically a couple of minutes for 3000 models.

import inspect as _inspect

_LIST_PARAMS = set(_inspect.signature(api.list_models).parameters)


def list_models_kwargs():
    """huggingface_hub >=1.0 removed `direction` and sorts descending by default.
    Older versions need direction=-1 to get the most-downloaded models first."""
    kw = {"limit": CONFIG["n_models"], "cardData": True, "sort": CONFIG["sort_by"]}
    if "direction" in _LIST_PARAMS:
        kw["direction"] = -1
    return kw


def stage_a_collect(force=False):
    if MODELS_JSONL.exists() and not force:
        print("Stage A cached, skipping. Delete", MODELS_JSONL.name, "to re-run.")
        return read_jsonl(MODELS_JSONL)

    rows, seen = [], set()
    t0 = time.time()
    it = api.list_models(**list_models_kwargs())
    for info in it:
        rec = normalize_model_info(info)
        if not rec["id"] or rec["id"] in seen:
            continue
        seen.add(rec["id"])
        rows.append(rec)
        if len(rows) % 500 == 0:
            print(f"  {len(rows)} models, {time.time() - t0:.0f}s")

    write_jsonl(MODELS_JSONL, rows)
    print(f"Stage A done: {len(rows)} models in {time.time() - t0:.0f}s")
    return rows


models = stage_a_collect()
df = pd.DataFrame(models)
print(df.shape)


def field_coverage(models):
    """If the Hub or the client version changes field names, this catches it
    before you spend an hour on Stage C."""
    n = len(models)
    checks = {
        "id": sum(1 for m in models if m.get("id")),
        "downloads": sum(1 for m in models if m.get("downloads") is not None),
        "license_declared": sum(1 for m in models if m.get("license_declared")),
        "base_models": sum(1 for m in models if m.get("base_models")),
        "has_card_data": sum(1 for m in models if m.get("has_card_data")),
    }
    print("\nField coverage:")
    for k, v in checks.items():
        flag = "  <-- CHECK THIS" if v == 0 else ""
        print(f"  {k:18s} {v:5d}/{n} ({v / n:.1%}){flag}")
    dls = [m.get("downloads") or 0 for m in models[:50]]
    if dls and dls[0] < dls[-1]:
        print("\n  WARNING: sample does not look sorted descending by downloads.")
    if checks["license_declared"] == 0 or checks["base_models"] == 0:
        print("\n  Print one raw object to inspect field names:")
        print("  info = next(iter(api.list_models(**list_models_kwargs()))); print(vars(info))")


field_coverage(models)
df.head(3)[["id", "downloads", "license_declared", "base_models"]]

In [ ]:
# Cell 5: STAGE B. Fetch metadata for declared parents not already in the sample.
# Needed to compare a child's license against its parent's.

def stage_b_parents(models, force=False):
    if PARENTS_JSONL.exists() and not force:
        print("Stage B cached, skipping.")
        return read_jsonl(PARENTS_JSONL)

    have = {m["id"] for m in models}
    wanted = set()
    for m in models:
        for parent in m["base_models"]:
            if parent not in have:
                wanted.add(parent)
    wanted = sorted(wanted)
    print(f"{len(wanted)} parent repos to fetch")

    rows, t0 = [], time.time()
    for i, pid in enumerate(wanted, 1):
        try:
            info = api.model_info(pid)
            rows.append(normalize_model_info(info))
        except (RepositoryNotFoundError, GatedRepoError) as exc:
            rows.append({"id": pid, "fetch_error": type(exc).__name__})
        except HfHubHTTPError as exc:
            rows.append({"id": pid, "fetch_error": f"http:{getattr(exc.response, 'status_code', '?')}"})
        except Exception as exc:  # keep the crawl alive, record the failure
            rows.append({"id": pid, "fetch_error": type(exc).__name__})
        if i % 100 == 0:
            print(f"  {i}/{len(wanted)}, {time.time() - t0:.0f}s")
        time.sleep(0.05)

    write_jsonl(PARENTS_JSONL, rows)
    print(f"Stage B done: {len(rows)} parents in {time.time() - t0:.0f}s")
    return rows


parents = stage_b_parents(models)

# Unified lookup: sample models plus fetched parents
LOOKUP = {m["id"]: m for m in models}
for p in parents:
    LOOKUP.setdefault(p["id"], p)
print("lookup size:", len(LOOKUP))

In [ ]:
# Cell 6: STAGE C. README bodies for the prose audit (Paper 2).
# The slow stage. 600 files is roughly 5 to 10 minutes.
# Sampling is stratified by download rank so the subset is not all head models.

def stage_c_readmes(models, force=False):
    if README_JSONL.exists() and not force:
        print("Stage C cached, skipping.")
        return read_jsonl(README_JSONL)

    ranked = sorted(
        [m for m in models if m.get("id")],
        key=lambda m: (m.get("downloads") or 0),
        reverse=True,
    )
    n = min(CONFIG["n_readme"], len(ranked))
    n_strata = 6
    per = max(1, n // n_strata)
    stratum_size = max(1, len(ranked) // n_strata)

    rng = random.Random(CONFIG["seed"])
    picked = []
    for s in range(n_strata):
        chunk = ranked[s * stratum_size:(s + 1) * stratum_size]
        if chunk:
            picked.extend(rng.sample(chunk, min(per, len(chunk))))
    picked = picked[:n]
    print(f"{len(picked)} READMEs to fetch")

    rows, t0 = [], time.time()
    for i, m in enumerate(picked, 1):
        rid = m["id"]
        rec = {"id": rid, "readme": None, "fetch_error": None}
        try:
            path = hf_hub_download(repo_id=rid, filename="README.md", repo_type="model", token=HF_TOKEN)
            with open(path, encoding="utf-8", errors="replace") as fh:
                rec["readme"] = fh.read()[: CONFIG["readme_max_chars"]]
        except (EntryNotFoundError, RepositoryNotFoundError, GatedRepoError) as exc:
            rec["fetch_error"] = type(exc).__name__
        except Exception as exc:
            rec["fetch_error"] = type(exc).__name__
        rows.append(rec)
        if i % 50 == 0:
            print(f"  {i}/{len(picked)}, {time.time() - t0:.0f}s")
        time.sleep(CONFIG["sleep_between_readmes"])

    write_jsonl(README_JSONL, rows)
    ok = sum(1 for r in rows if r["readme"])
    print(f"Stage C done: {ok}/{len(rows)} READMEs retrieved in {time.time() - t0:.0f}s")
    return rows


readmes = stage_c_readmes(models)
README_BY_ID = {r["id"]: r for r in readmes}

In [ ]:
# Cell 7: license normalization.
# Rank is ordinal only. It encodes "how freely can a downstream user act",
# not legal equivalence. Document this in the paper as a coding decision.

PERMISSIVE = {"mit", "apache-2.0", "bsd", "bsd-2-clause", "bsd-3-clause", "cc0-1.0",
              "cc-by-4.0", "cc-by-3.0", "unlicense", "isc", "artistic-2.0", "wtfpl",
              "postgresql", "zlib", "mpl-2.0"}
COPYLEFT = {"gpl", "gpl-2.0", "gpl-3.0", "agpl-3.0", "lgpl", "lgpl-2.1", "lgpl-3.0",
            "cc-by-sa-4.0", "cc-by-sa-3.0", "osl-3.0", "epl-2.0"}
RESTRICTED_PREFIXES = ("openrail", "creativeml-openrail", "bigscience-openrail",
                       "bigcode-openrail", "llama", "gemma", "cc-by-nc", "cc-by-nd",
                       "apple-a", "deepfloyd")
RESTRICTED_EXACT = {"other", "proprietary", "unknown", "bigscience-bloom-rail-1.0",
                    "cc-by-nc-4.0", "cc-by-nc-sa-4.0", "cc-by-nc-nd-4.0"}

RANK = {"permissive": 3, "copyleft": 2, "restricted": 1, "undeclared": 0}


def license_family(lic):
    if not lic or not isinstance(lic, str):
        return "undeclared"
    key = lic.strip().lower()
    if key in PERMISSIVE:
        return "permissive"
    if key in COPYLEFT or any(key.startswith(c) for c in ("gpl", "agpl", "lgpl", "cc-by-sa")):
        return "copyleft"
    if key in RESTRICTED_EXACT or key.startswith(RESTRICTED_PREFIXES):
        return "restricted"
    return "restricted"  # unrecognized strings are conservatively non-permissive

In [ ]:
# Cell 8: ANALYSIS 1 (Paper 1a). Provenance completeness.
# How often is base_model declared at all? Everything downstream depends on it.

work = df.copy()
work["downloads"] = work["downloads"].fillna(0)
work["has_base_model"] = work["base_models"].apply(lambda b: len(b) > 0)
work["decile"] = pd.qcut(work["downloads"].rank(method="first", ascending=False),
                         10, labels=[f"D{i}" for i in range(1, 11)])
work["org"] = work["id"].str.split("/").str[0]

completeness = work.groupby("decile", observed=True)["has_base_model"].agg(["mean", "count"])
completeness.columns = ["declared_rate", "n"]
print("Provenance completeness by download decile (D1 = most downloaded)")
print(completeness.round(3))

overall = work["has_base_model"].mean()
print(f"\nOverall base_model declaration rate: {overall:.1%} of {len(work)} models")
print("\nRelation kinds among models that declare a parent:")
print(work.loc[work["has_base_model"], "relation_kind"].value_counts())

In [ ]:
# Cell 9: ANALYSIS 2 (Paper 1b). License inheritance across one derivation hop.

records = []
for m in models:
    child_lic = m.get("license_declared")
    child_fam = license_family(child_lic)
    for parent_id in m["base_models"]:
        p = LOOKUP.get(parent_id)
        if not p or p.get("fetch_error"):
            outcome = "parent_unresolved"
            parent_fam, parent_lic = None, None
        else:
            parent_lic = p.get("license_declared")
            parent_fam = license_family(parent_lic)
            if parent_fam == "undeclared":
                outcome = "parent_undeclared"
            elif child_fam == "undeclared":
                outcome = "dropped"
            elif (child_lic or "").lower() == (parent_lic or "").lower():
                outcome = "identical"
            elif RANK[child_fam] > RANK[parent_fam]:
                outcome = "loosened"
            elif RANK[child_fam] < RANK[parent_fam]:
                outcome = "tightened"
            else:
                outcome = "same_family_different_license"
        records.append({
            "child": m["id"], "parent": parent_id,
            "child_license": child_lic, "parent_license": parent_lic,
            "child_family": child_fam, "parent_family": parent_fam,
            "relation_kind": m["relation_kind"],
            "child_downloads": m.get("downloads") or 0,
            "outcome": outcome,
        })

edges = pd.DataFrame(records)
print(f"{len(edges)} parent-child edges from {edges['child'].nunique()} child models\n")
print(edges["outcome"].value_counts())

resolved = edges[~edges["outcome"].isin(["parent_unresolved", "parent_undeclared"])]
if len(resolved):
    print(f"\nAmong {len(resolved)} resolvable edges:")
    print((resolved["outcome"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")

    restrictive_parents = resolved[resolved["parent_family"] == "restricted"]
    if len(restrictive_parents):
        leak = restrictive_parents["outcome"].isin(["loosened", "dropped"]).mean()
        print(f"\nHEADLINE: of {len(restrictive_parents)} derivatives of restrictively "
              f"licensed parents, {leak:.1%} either loosen or drop the license.")

edges.to_csv(CACHE / "license_edges.csv", index=False)

In [ ]:
# Cell 10: ANALYSIS 3 (Paper 2). Frontmatter versus prose contradictions.

FRONTMATTER_RE = re.compile(r"^---\s*\n.*?\n---\s*\n", re.DOTALL)
RESTRICTION_RE = re.compile(
    r"(non[-\s]?commercial|research\s+(?:use\s+)?only|not\s+(?:be\s+)?used\s+for\s+commercial"
    r"|acceptable\s+use\s+policy|use\s+restrictions|may\s+not\s+be\s+used\s+to)",
    re.IGNORECASE)
LICENSE_MENTION_RE = re.compile(
    r"\b(mit\s+license|apache\s*2(\.0)?|gpl|agpl|cc[-\s]?by(?:[-\s]?nc)?(?:[-\s]?sa)?"
    r"|openrail|creativeml|llama\s*[\d.]*\s*(?:community\s*)?license|gemma\s+terms)\b",
    re.IGNORECASE)
BASE_MENTION_RE = re.compile(
    r"(?:fine[-\s]?tuned?|finetuned?|based|built|trained|distilled|quantiz(?:ed|ation))"
    r"[^.\n]{0,60}?\b(?:of|on|from|upon)\s+\[?([A-Za-z0-9][\w.\-]{1,40}/[\w.\-]{1,58}[\w\-])\]?",
    re.IGNORECASE)


def strip_frontmatter(text):
    return FRONTMATTER_RE.sub("", text or "", count=1)


rows = []
for m in models:
    rec = README_BY_ID.get(m["id"])
    if not rec or not rec.get("readme"):
        continue
    body = strip_frontmatter(rec["readme"])
    fam = license_family(m.get("license_declared"))

    prose_restriction = bool(RESTRICTION_RE.search(body))
    prose_license = bool(LICENSE_MENTION_RE.search(body))
    prose_bases = {b.strip().rstrip(").,") for b in BASE_MENTION_RE.findall(body)}
    declared_bases = {b.lower() for b in m["base_models"]}
    prose_bases_l = {b.lower() for b in prose_bases}

    c1 = fam == "permissive" and prose_restriction
    c2 = fam == "undeclared" and prose_license
    c3 = (not declared_bases) and bool(prose_bases_l)
    c4 = bool(declared_bases) and bool(prose_bases_l) and not (declared_bases & prose_bases_l)

    rows.append({
        "id": m["id"],
        "downloads": m.get("downloads") or 0,
        "license_declared": m.get("license_declared"),
        "license_family": fam,
        "declared_base": ";".join(sorted(declared_bases)) or None,
        "prose_base": ";".join(sorted(prose_bases_l)) or None,
        "prose_restriction_language": prose_restriction,
        "C1_permissive_tag_restrictive_prose": c1,
        "C2_no_tag_license_in_prose": c2,
        "C3_no_tag_base_in_prose": c3,
        "C4_tag_and_prose_base_disagree": c4,
        "any_contradiction": bool(c1 or c2 or c3 or c4),
        "readme_chars": len(body),
    })

prose = pd.DataFrame(rows)
print(f"Prose audit over {len(prose)} cards with a retrievable README\n")
for col in ["C1_permissive_tag_restrictive_prose", "C2_no_tag_license_in_prose",
            "C3_no_tag_base_in_prose", "C4_tag_and_prose_base_disagree"]:
    print(f"  {col}: {prose[col].sum()} ({prose[col].mean():.1%})")
print(f"\n  ANY contradiction: {prose['any_contradiction'].sum()} ({prose['any_contradiction'].mean():.1%})")

# Recoverability: when the tag is missing, how often does the prose supply it?
missing_tag = prose[prose["declared_base"].isna()]
if len(missing_tag):
    print(f"\nOf {len(missing_tag)} cards with no base_model tag, "
          f"{missing_tag['prose_base'].notna().mean():.1%} name a parent in the prose.")

prose.to_csv(CACHE / "prose_contradictions.csv", index=False)

In [ ]:
# CELL 10b. Replaces Cell 10. Runs entirely off the existing cache, no re-crawl.
#
# Two defects in the first version, both found by checking against real cards:
#   1. Cards name parents as markdown links whose visible text omits the org,
#      e.g. "a fine-tuned version of [vit-base-patch16-384](https://huggingface.co/google/...)".
#      Parents are now read from huggingface.co URLs as well as bare org/name tokens.
#   2. "can be fine-tuned", "in fine-tuned settings" are not provenance claims.
#      Extraction is now sentence-scoped: a repo id only counts as a declared parent
#      if it appears in the same sentence as a provenance trigger.
#
# Bare org/name tokens are validated against the set of organizations actually
# observed on the Hub in this crawl, which removes false positives such as
# "building/utilizing". Dataset, space, blog, and paper URLs are excluded.

import json
import re
import pandas as pd

models = {m["id"]: m for m in read_jsonl(MODELS_JSONL)}
parents_all = {p["id"]: p for p in read_jsonl(PARENTS_JSONL)}
readme_by_id = {r["id"]: r for r in read_jsonl(README_JSONL)}

KNOWN_ORGS = {k.split("/")[0].lower() for k in list(models) + list(parents_all) if "/" in k}
print(f"{len(KNOWN_ORGS)} known organizations available for id validation")

FRONT_RE = re.compile(r"^---\s*\r?\n.*?\r?\n---\s*\r?\n", re.DOTALL)  # \r\n tolerant
HF_URL_RE = re.compile(
    r"huggingface\.co/(?!datasets/|spaces/|blog/|docs/|papers/|collections/)"
    r"([A-Za-z0-9][\w.\-]{1,40}/[\w.\-]{1,58}[\w\-])", re.IGNORECASE)
BARE_ID_RE = re.compile(r"(?<![\w/.])([A-Za-z0-9][\w.\-]{1,40})/([\w.\-]{1,58}[\w\-])(?![\w/])")
PROVENANCE_RE = re.compile(
    r"(fine[-\s]?tun\w*\s+(?:version\s+)?(?:of|from)|based\s+(?:on|upon)|derived\s+from"
    r"|distilled\s+from|quantiz\w+\s+(?:version\s+)?(?:of|from)|adapter\s+(?:of|for)"
    r"|merge\s+of|continued\s+pre-?training\s+(?:of|from)|initiali[sz]ed\s+from"
    r"|checkpoint\s+of|built\s+(?:on|upon)|starting\s+from|trained\s+(?:on\s+top\s+of|from))",
    re.IGNORECASE)


def split_sentences(text):
    return re.split(r"(?<=[.!?])\s+|\n{2,}|\n[-*#]\s*", text)


def extract_repo_ids(segment):
    """URLs are trusted outright. Bare tokens only if the org was seen on the Hub."""
    out = {m.group(1) for m in HF_URL_RE.finditer(segment)}
    for m in BARE_ID_RE.finditer(segment):
        if m.group(1).lower() in KNOWN_ORGS:
            out.add(f"{m.group(1)}/{m.group(2)}")
    return {o.rstrip("/.,);") for o in out}


records = []
for mid, m in models.items():
    rec = readme_by_id.get(mid)
    if not rec or not rec.get("readme"):
        continue
    body = FRONT_RE.sub("", rec["readme"], count=1)
    own = {mid.lower(), mid.split("/")[-1].lower()}

    prose_parents = set()
    for sent in split_sentences(body):
        if PROVENANCE_RE.search(sent):
            prose_parents |= {r for r in (x.lower() for x in extract_repo_ids(sent)) if r not in own}

    declared = {b.lower() for b in (m.get("base_models") or [])}
    short = lambda s: {x.split("/")[-1] for x in s}
    agree = bool(declared & prose_parents) or bool(short(declared) & short(prose_parents))

    records.append({
        "id": mid,
        "downloads": m.get("downloads") or 0,
        "license_declared": m.get("license_declared"),
        "declared_base": ";".join(sorted(declared)) or None,
        "prose_base": ";".join(sorted(prose_parents)) or None,
        "has_declared": bool(declared),
        "has_prose": bool(prose_parents),
        "agree": agree,
        "readme_chars": len(body),
    })

pp = pd.DataFrame(records)
both = pp[pp.has_declared & pp.has_prose]
untagged = (~pp.has_declared).sum()

print(f"\nCards audited: {len(pp)}")
print(f"  tag declares a parent : {pp.has_declared.sum():4d} ({pp.has_declared.mean():.1%})")
print(f"  prose names a parent  : {pp.has_prose.sum():4d} ({pp.has_prose.mean():.1%})")
print(f"\nBOTH present : {len(both):4d}   agree {both.agree.sum()} ({both.agree.mean():.1%})"
      f" | DISAGREE {(~both.agree).sum()} ({1 - both.agree.mean():.1%})")
print(f"RECOVERABLE  : {(~pp.has_declared & pp.has_prose).sum():4d} of {untagged} untagged cards"
      f" ({(~pp.has_declared & pp.has_prose).sum() / untagged:.1%})")
print(f"TAG ONLY     : {(pp.has_declared & ~pp.has_prose).sum():4d}"
      f" ({(pp.has_declared & ~pp.has_prose).sum() / pp.has_declared.sum():.1%} of tagged cards)")
print(f"SILENT       : {(~pp.has_declared & ~pp.has_prose).sum():4d}"
      f" ({(~pp.has_declared & ~pp.has_prose).mean():.1%})")

pp.to_csv(CACHE / "prose_parents_fixed.csv", index=False)

# Validation set: every disagreement, plus samples of the other two cells.
# Disagreements are few enough to code exhaustively, which is stronger than sampling.
disagree = both[~both.agree].copy()
recoverable = pp[~pp.has_declared & pp.has_prose]
agreeing = both[both.agree]

val = pd.concat([
    disagree.assign(cell="disagree"),
    recoverable.sample(min(20, len(recoverable)), random_state=CONFIG["seed"]).assign(cell="recoverable"),
    agreeing.sample(min(20, len(agreeing)), random_state=CONFIG["seed"]).assign(cell="agree"),
])
val["card_url"] = "https://huggingface.co/" + val["id"]
val["human_verdict"] = ""   # true_positive / false_positive / unclear
val["human_note"] = ""
val = val.sample(frac=1.0, random_state=CONFIG["seed"])   # shuffle so coding is blind
val.to_csv(CACHE / "validation_parents_TO_CODE.csv", index=False)
print(f"\nWrote {len(val)} rows to validation_parents_TO_CODE.csv "
      f"({len(disagree)} disagreements coded exhaustively)")


In [ ]:
# Cell 11: MANUAL VALIDATION SET.
# Regex-derived flags are not a finding until a human checks them. Code this
# sample yourself, report the agreement rate in the paper, and keep the file.

val_rng = random.Random(CONFIG["seed"] + 1)
flagged = prose[prose["any_contradiction"]]
clean = prose[~prose["any_contradiction"]]

n_flag = min(30, len(flagged))
n_clean = min(15, len(clean))
sample = pd.concat([
    flagged.sample(n_flag, random_state=CONFIG["seed"]) if n_flag else flagged,
    clean.sample(n_clean, random_state=CONFIG["seed"]) if n_clean else clean,
])
sample = sample.sample(frac=1.0, random_state=CONFIG["seed"])  # shuffle so coding is blind
sample["human_verdict"] = ""        # fill in: true_positive / false_positive / unclear
sample["human_note"] = ""
sample["card_url"] = "https://huggingface.co/" + sample["id"]

val_path = CACHE / "validation_sample_TO_CODE.csv"
sample.to_csv(val_path, index=False)
print(f"Wrote {len(sample)} rows to {val_path}")
print("Open each card_url, fill human_verdict, then compute agreement below.")

In [ ]:
# Cell 12: agreement, once the validation sheet is filled in.

def agreement_report(path=None):
    path = path or (CACHE / "validation_sample_CODED.csv")
    if not pathlib.Path(path).exists():
        print(f"No coded file at {path} yet. Code the sample first.")
        return None
    coded = pd.read_csv(path)
    coded = coded[coded["human_verdict"].astype(str).str.strip() != ""]
    if not len(coded):
        print("Coded file has no verdicts yet.")
        return None
    tp = (coded["human_verdict"] == "true_positive").sum()
    fp = (coded["human_verdict"] == "false_positive").sum()
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    print(f"Coded {len(coded)} cards. Flag precision: {prec:.1%} ({tp} TP, {fp} FP)")
    return coded


_ = agreement_report()

In [ ]:
# Cell 13: figures

FIGDIR = CACHE / "figures"
FIGDIR.mkdir(exist_ok=True)
matplotlib.rcParams.update({"figure.dpi": 150, "font.size": 9, "savefig.bbox": "tight"})

# Figure 1: provenance completeness by download decile
fig, ax = plt.subplots(figsize=(5.2, 2.8))
ax.bar(completeness.index.astype(str), completeness["declared_rate"] * 100)
ax.set_xlabel("Download decile (D1 = most downloaded)")
ax.set_ylabel("base_model declared (%)")
ax.set_title("Provenance completeness")
fig.savefig(FIGDIR / "fig1_provenance_completeness.png")
plt.close(fig)

# Figure 2: license inheritance outcomes
if len(resolved):
    counts = resolved["outcome"].value_counts()
    fig, ax = plt.subplots(figsize=(5.2, 2.8))
    ax.barh(counts.index[::-1].astype(str), counts.values[::-1])
    ax.set_xlabel("Parent-child edges")
    ax.set_title("License inheritance across one derivation hop")
    fig.savefig(FIGDIR / "fig2_license_inheritance.png")
    plt.close(fig)

# Figure 3: contradiction types
cols = ["C1_permissive_tag_restrictive_prose", "C2_no_tag_license_in_prose",
        "C3_no_tag_base_in_prose", "C4_tag_and_prose_base_disagree"]
fig, ax = plt.subplots(figsize=(5.2, 2.8))
ax.bar([c.split("_")[0] for c in cols], [prose[c].mean() * 100 for c in cols])
ax.set_ylabel("Cards affected (%)")
ax.set_title("Frontmatter versus prose contradictions")
fig.savefig(FIGDIR / "fig3_contradictions.png")
plt.close(fig)

print("Figures written to", FIGDIR)
for f in sorted(FIGDIR.glob("*.png")):
    print(" ", f.name)

In [ ]:
# Cell 14: results manifest. Attach this to the paper and the repo.

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


manifest = {
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "config": CONFIG,
    "environment": {
        "python": platform.python_version(),
        "huggingface_hub": huggingface_hub.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "counts": {
        "models_sampled": int(len(df)),
        "parents_fetched": int(len(parents)),
        "readmes_retrieved": int(sum(1 for r in readmes if r.get("readme"))),
        "edges_total": int(len(edges)),
        "edges_resolvable": int(len(resolved)),
        "prose_cards_audited": int(len(prose)),
    },
    "headline_rates": {
        "base_model_declaration_rate": float(overall),
        "any_contradiction_rate": float(prose["any_contradiction"].mean()) if len(prose) else None,
    },
    "artifacts": {
        p.name: {"bytes": p.stat().st_size, "sha256_16": file_hash(p)}
        for p in sorted(CACHE.glob("*.jsonl")) + sorted(CACHE.glob("*.csv"))
    },
}

with open(MANIFEST, "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2)

print(json.dumps({k: manifest[k] for k in ["counts", "headline_rates"]}, indent=2))
print("\nManifest:", MANIFEST)

In [ ]:
# CELL 14b. Run AFTER Cell 10b. Rewrites manifest.json so the recorded headline
# figures come from the corrected prose analysis rather than the superseded Cell 10.
# Cell 14 runs before 10b in sheet order, so without this the manifest still
# reports the old 2.2% contradiction rate, which is not what the paper claims.

import json

edges_resolvable = edges[~edges["outcome"].isin(["parent_unresolved", "parent_undeclared"])]
restricted_parents = edges_resolvable[edges_resolvable["parent_family"] == "restricted"]
both_channels = pp[pp.has_declared & pp.has_prose]

with open(MANIFEST, encoding="utf-8") as fh:
    manifest = json.load(fh)

manifest["analysis_version"] = "v2 (cell 10b prose extraction)"
manifest["superseded"] = {
    "cell_10_any_contradiction_rate": manifest.get("headline_rates", {}).get("any_contradiction_rate"),
    "reason": "Cell 10 missed markdown-link parents and counted non-provenance uses of 'fine-tuned'.",
}
manifest["headline_rates"] = {
    "base_model_declaration_rate": float(df["base_models"].apply(len).gt(0).mean()),
    "license_identical_rate": float((edges_resolvable["outcome"] == "identical").mean()),
    "restricted_parent_leak_rate": float(
        restricted_parents["outcome"].isin(["loosened", "dropped"]).mean()),
    "prose_names_parent_rate": float(pp.has_prose.mean()),
    "both_channels_disagree_rate": float(1 - both_channels.agree.mean()) if len(both_channels) else None,
    "silent_rate": float((~pp.has_declared & ~pp.has_prose).mean()),
}
manifest["counts"].update({
    "edges_restricted_parent": int(len(restricted_parents)),
    "cards_both_channels": int(len(both_channels)),
    "cards_silent": int((~pp.has_declared & ~pp.has_prose).sum()),
})

with open(MANIFEST, "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2)

print(json.dumps(manifest["headline_rates"], indent=2))
print("\nLeak rate by relation type (restricted parents only):")
for kind, grp in restricted_parents.groupby("relation_kind"):
    leak = grp["outcome"].isin(["loosened", "dropped"]).mean()
    print(f"  {kind:10s} n={len(grp):4d}  leak={leak:.1%}")


In [ ]:
# Cell 15: zip the cache so a Colab session loss costs you nothing.

import shutil
archive = shutil.make_archive("/content/hf_gov_cache_export", "zip", root_dir=str(CACHE))
print("Archive:", archive)
# In Colab: from google.colab import files; files.download(archive)